# EpistemicOps: GRPO Training via Unsloth & HuggingFace TRL

This notebook demonstrates how to load an OpenEnv environment (EpistemicOps), initialize an Unsloth 4-bit Llama-3.1 model, and train it using GRPO to handle temporal drift and Socratic oversight.

In [ ]:
!pip install -q unsloth trl openenv httpx

In [ ]:
import torch
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
import httpx
import asyncio
import os

# Set API endpoint if running remotely
EPISTEMICOPS_ENV_URL = os.getenv("EPISTEMICOPS_ENV_URL", "http://localhost:8000")

## 1. Load Unsloth Model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct",
    max_seq_length=8192,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=16
)

# Apply LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth"
)

## 2. Define OpenEnv Reward Wrapper

In [ ]:
async def evaluate_trajectory(completion):
    """
    Simulate the trajectory through the EpistemicOps FastAPI server
    and return the computed total reward.
    """
    # Note: A real implementation traces the completion text step-by-step
    # against the environment. For demonstration, we simulate fetching state.
    try:
        async with httpx.AsyncClient() as client:
            # Reset scenario
            resp = await client.post(f"{EPISTEMICOPS_ENV_URL}/reset", json={"scenario_id": "cascading_incident"})
            # Return a mock reward scalar to represent trajectory success
            return 0.75 
    except:
        # If server is not running
        return 0.0

def epistemicops_reward(completions, **kwargs):
    # Run async evaluation synchronously for batch
    loop = asyncio.get_event_loop()
    rewards = []
    for completion in completions:
        r = loop.run_until_complete(evaluate_trajectory(completion))
        rewards.append(r)
    return rewards

## 3. Configure and Train

In [ ]:
training_args = GRPOConfig(
    output_dir="./checkpoints/primary_agent",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    kl_coef=0.1,
    temperature=0.8,
    logging_steps=5,
    save_steps=50,
    report_to="none"
)

# Dummy dataset of initial prompts
from datasets import Dataset
dataset = Dataset.from_list([{"prompt": "You are the Primary Agent (Student). Era 1 begins."} for _ in range(50)])

trainer = GRPOTrainer(
    model=model,
    reward_funcs=epistemicops_reward,
    args=training_args,
    train_dataset=dataset,
)

trainer.train()